# M07 — 串流、可觀測與整合專案（Capstone）

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

三條主線：
1. **串流**：用同一張圖示範 `stream_mode` 的 `"values"` / `"updates"` / `"messages"` 三種輸出。
2. **可觀測**：說明如何用環境變數開啟 LangSmith 追蹤，以及 UI 上能看到什麼（不需真的連線）。
3. **Capstone**：把 RAG（第一冊 M05）＋工具（M04）＋記憶 checkpointer（第二冊 M04）
   ＋人介入 interrupt（第二冊 M05）整合進**同一張圖**，用 `stream` 跑完一個完整情境。

## 1. 環境準備

跟前面模組一樣，透過 `_shared/course_utils.py` 取得供應商無關的模型與 embeddings。
這一格沒有可見輸出，成功代表 `model` / `embeddings` 已就緒。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, get_embeddings, load_env

load_env()
model = get_model()
embeddings = get_embeddings()

## 2. 先做一張最小的圖，用來示範三種串流

串流的差異跟圖的內容無關，所以我們先搭一張**最小**的兩節點圖：
`tell_topic` 寫入一個主題，`write` 讓模型針對主題寫一句話。
重點是這張圖會更新 state、也會呼叫模型——剛好夠示範三種 `stream_mode`。

這一格只是建圖與 compile，沒有可見輸出。

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class DemoState(TypedDict):
    topic: str
    answer: str


def tell_topic(state: DemoState) -> dict:
    # Just records the topic into state (an "update" with no model call).
    return {"topic": state["topic"]}


def write(state: DemoState) -> dict:
    # Calls the model; this is where token-level streaming happens.
    ai = model.invoke(f"用一句話介紹「{state['topic']}」")
    return {"answer": ai.content}


demo_builder = StateGraph(DemoState)
demo_builder.add_node("tell_topic", tell_topic)
demo_builder.add_node("write", write)
demo_builder.add_edge(START, "tell_topic")
demo_builder.add_edge("tell_topic", "write")
demo_builder.add_edge("write", END)
demo_graph = demo_builder.compile()

# Print the graph as ASCII so you can see its shape (from M01).
print(demo_graph.get_graph().draw_ascii())
# Expected output (大致如下):
# +-----------+
# | __start__ |
# +-----------+
#       *
# +------------+
# | tell_topic |
# +------------+
#       *
#   +-------+
#   | write |
#   +-------+
#       *
#  +---------+
#  | __end__ |
#  +---------+

## 3. `stream_mode="values"`：看每步**之後的完整 state**

每個 chunk 是一張**完整快照**——整個 state dict，key 都在，值隨節點推進而變多/變新。
適合除錯：你能看到 state 一步步演化成什麼樣。

預期會印出數個 chunk：初始輸入、`tell_topic` 後（多了 topic）、`write` 後（多了 answer）。

In [ ]:
demo_inputs = {"topic": "向量資料庫", "answer": ""}

for chunk in demo_graph.stream(demo_inputs, stream_mode="values"):
    print(chunk)
# Expected output (大致如下，每行是一個完整 state 快照):
# {'topic': '向量資料庫', 'answer': ''}
# {'topic': '向量資料庫', 'answer': ''}
# {'topic': '向量資料庫', 'answer': '向量資料庫是把資料以向量形式儲存、用相似度搜尋的資料庫。'}

## 4. `stream_mode="updates"`：看每步**只回傳的增量**

每個 chunk 形如 `{節點名: 那個節點 return 的 dict}`。
你一眼就知道「是哪個節點、改了什麼」——做進度提示（「正在檢索…」「正在生成…」）最好用。

對比上一格：`values` 給整張 state，`updates` 只給「剛剛這一步的差異」。

In [ ]:
for chunk in demo_graph.stream(demo_inputs, stream_mode="updates"):
    print(chunk)
# Expected output (大致如下，每行是一個節點的增量):
# {'tell_topic': {'topic': '向量資料庫'}}
# {'write': {'answer': '向量資料庫是把資料以向量形式儲存、用相似度搜尋的資料庫。'}}

## 5. `stream_mode="messages"`：看 token 級的**逐字輸出**

每個 chunk 是 `(token, metadata)` 的 tuple。`token` 是一個 message chunk，
文字在 `token.content`；`metadata` 告訴你這個 token 來自哪個節點。
這就是「打字機效果」的來源——模型一個字一個字吐，你就一個字一個字顯示。

⚠️ 陷阱：一定要解包成 `for token, meta in ...`，且取文字用 `token.content`；
直接 `print(chunk)` 會印出一坨 tuple。

In [ ]:
print("打字機輸出：", end="")
for token, meta in demo_graph.stream(demo_inputs, stream_mode="messages"):
    # Only the `write` node calls the model, so tokens come from there.
    print(token.content, end="", flush=True)
print()
# Expected output (大致如下，逐字串出最後組成一句話):
# 打字機輸出：向量資料庫是把資料以向量形式儲存、用相似度搜尋的資料庫。

## 6. 可觀測性：開啟 LangSmith 追蹤

LangSmith 的好處是**不用改程式碼**：只要設好環境變數，所有 LangChain / LangGraph
的執行就會自動上傳追蹤。你照常 `invoke` / `stream` 即可。

在終端機（或 `.env`）設定：

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=ls__你的金鑰
export LANGSMITH_PROJECT=langgraph-capstone   # 選填，分組用
```

設好後到 https://smith.langchain.com，每次執行會變成一條 **trace**（一棵樹），你能看到：

- 圖跑了哪些節點、各自的**輸入/輸出**與耗時。
- 每次模型呼叫的**完整 prompt、回應、token 數、成本**。
- 工具呼叫的參數與回傳值；出錯時是哪一步、丟了什麼 exception。

下面這格只示範「如何用程式碼檢查是否已開啟」，**不需要真的連線**。

In [ ]:
import os

tracing_on = os.environ.get("LANGSMITH_TRACING", "").lower() == "true"
has_key = bool(os.environ.get("LANGSMITH_API_KEY"))

print("LANGSMITH_TRACING 已開啟:", tracing_on)
print("LANGSMITH_API_KEY 已設定:", has_key)
if tracing_on and has_key:
    print("=> 接下來的每次 invoke/stream 都會自動出現在 LangSmith UI")
else:
    print("=> 尚未開啟追蹤；設好上面兩個環境變數即可，不必改任何程式碼")
# Expected output (在未設定的環境下，大致如下):
# LANGSMITH_TRACING 已開啟: False
# LANGSMITH_API_KEY 已設定: False
# => 尚未開啟追蹤；設好上面兩個環境變數即可，不必改任何程式碼

## 7. Capstone（一）：準備 RAG 的向量庫

從這裡開始整合全課程。先用第一冊 M05 的做法，把一小段「公司知識」建成可檢索的向量庫。
真實情境會切大量文件；這裡用三句話示意即可。

這一格建好 `retriever`，沒有可見輸出。

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# A tiny private knowledge base the base model could never know on its own.
docs = [
    "本公司退款政策：商品到貨 7 天內可申請全額退款，需保持包裝完整。",
    "本公司客服時間為週一至週五 09:00-18:00，例假日不提供電話客服。",
    "VIP 會員享有免運費與專屬折扣碼 VIP2026，每季可使用一次。",
]

store = InMemoryVectorStore(embeddings)
store.add_texts(docs)
retriever = store.as_retriever(search_kwargs={"k": 2})

## 8. Capstone（二）：準備一個工具

用第一冊 M04 的 `@tool` 定義一個工具，給 agent 節點用。
這裡做一個「下單」工具——故意設計成「會改變外部狀態」的危險操作，
好讓後面的**人工核准**節點有意義（真要下單前先問人）。

這一格只定義工具，沒有可見輸出。

In [ ]:
from langchain.tools import tool


@tool
def place_order(item: str, quantity: int) -> str:
    """Place an order for a given item and quantity. This changes external state."""
    # A real tool would call an order API here; we return a fake confirmation.
    return f"已下單：{item} x {quantity}，訂單編號 ORD-2026-001"


order_tools = [place_order]
tools_by_name = {t.name: t for t in order_tools}
model_with_tools = model.bind_tools(order_tools)

## 9. Capstone（三）：設計共用的 State

`State` 是把所有零件串起來的共用資料結構，它要同時裝得下：

- `messages`：對話歷史（用 `add_messages` reducer 累加，來自第二冊 M02）。
- `context`：RAG 檢索到的文字。
- `approved`：人工核准的結果。

用 `MessagesState` 也可以，但這裡明確列出三個欄位，資料流更清楚。

這一格只定義型別，沒有可見輸出。

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages


class AssistantState(TypedDict):
    messages: Annotated[list, add_messages]
    context: str
    approved: bool

## 10. Capstone（四）：定義四個節點

對照 README 的資料流：`retrieve → agent → human_approve → END`。

- `retrieve`：拿最後一則使用者訊息去檢索，把結果寫進 `context`（第一冊 M05）。
- `agent`：把 `context` 當系統提示餵給綁了工具的模型，讓它決定要不要下單（M04）。
- `human_approve`：用 `interrupt` 暫停，等人核准後才真正執行工具（第二冊 M05）。
- 我們不另外做 END 節點，`human_approve` 完直接收尾。

這一格只定義函式，沒有可見輸出。

In [ ]:
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.types import interrupt


def retrieve(state: AssistantState) -> dict:
    # Use the latest human message as the retrieval query.
    last_user = state["messages"][-1].content
    found = retriever.invoke(last_user)
    context = "\n".join(d.page_content for d in found)
    return {"context": context}


def agent(state: AssistantState) -> dict:
    # Feed retrieved context as a system message, then let the model decide.
    sys_msg = SystemMessage(
        "你是購物助理。可用知識如下，回答時優先參考：\n" + state["context"]
    )
    ai = model_with_tools.invoke([sys_msg, *state["messages"]])
    return {"messages": [ai]}


def human_approve(state: AssistantState) -> dict:
    last_ai = state["messages"][-1]

    # No tool requested -> nothing dangerous to approve, just pass through.
    if not last_ai.tool_calls:
        return {"approved": True}

    # Pause the graph and ask a human to approve the pending tool call.
    call = last_ai.tool_calls[0]
    decision = interrupt(
        {"question": f"是否核准執行 {call['name']}({call['args']})?"}
    )

    # On resume, `decision` carries whatever the human passed via Command(resume=...).
    if decision == "yes":
        result = tools_by_name[call["name"]].invoke(call["args"])
        tool_msg = ToolMessage(content=str(result), tool_call_id=call["id"])
        return {"approved": True, "messages": [tool_msg]}
    return {"approved": False}

## 11. Capstone（五）：組裝圖並掛上 checkpointer

把四個節點接成一條線，並在 `compile` 掛上 `InMemorySaver`（第二冊 M04）。

⚠️ 陷阱：`interrupt` 需要 checkpointer 才能保存「暫停點」、之後才能 resume。
沒掛 checkpointer，人介入節點會無法續跑。

這一格輸出圖的 ASCII 結構。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(AssistantState)
builder.add_node("retrieve", retrieve)
builder.add_node("agent", agent)
builder.add_node("human_approve", human_approve)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "agent")
builder.add_edge("agent", "human_approve")
builder.add_edge("human_approve", END)

# checkpointer is REQUIRED for interrupt() + resume to work.
assistant = builder.compile(checkpointer=InMemorySaver())

print(assistant.get_graph().draw_ascii())
# Expected output (大致如下):
# +-----------+
# | __start__ |
# +-----------+
#       *
#  +----------+
#  | retrieve |
#  +----------+
#       *
#   +-------+
#   | agent |
#   +-------+
#       *
# +---------------+
# | human_approve |
# +---------------+
#       *
#  +---------+
#  | __end__ |
#  +---------+

## 12. Capstone（六）：用 `stream` 跑到「停在核准點」

帶一個含 `thread_id` 的 config（checkpointer 靠它認得是同一段對話）。
用 `stream_mode="updates"` 跑，我們會逐步看到 `retrieve` → `agent` 的增量，
最後在 `human_approve` 觸發 `interrupt` 而**暫停**。

暫停時，串流會吐出一個帶 `__interrupt__` 的 chunk，裡面就是我們要問人的問題。

In [ ]:
config = {"configurable": {"thread_id": "capstone-1"}}
user_turn = {"messages": [HumanMessage("幫我下單 2 個 VIP 限定杯子")]}

print("=== 第一段：跑到核准點前 ===")
for chunk in assistant.stream(user_turn, config, stream_mode="updates"):
    print(chunk)
# Expected output (大致如下):
# === 第一段：跑到核准點前 ===
# {'retrieve': {'context': 'VIP 會員享有免運費...\n本公司退款政策...'}}
# {'agent': {'messages': [AIMessage(content='', tool_calls=[{'name': 'place_order', ...}])]}}
# {'__interrupt__': (Interrupt(value={'question': '是否核准執行 place_order({...})?'}),)}

## 13. Capstone（七）：人核准後續跑

用 `Command(resume="yes")` 把人的決定送回去，圖會從 `interrupt` 那一點繼續跑：
執行工具、回填 `ToolMessage`、收尾。注意 config 要用**同一個 `thread_id`**，
否則 checkpointer 找不到剛剛暫停的那張圖。

換 `resume="no"` 就會走「不核准、不下單」那條路，可自行試。

In [ ]:
from langgraph.types import Command

print("=== 第二段：核准後續跑 ===")
for chunk in assistant.stream(Command(resume="yes"), config, stream_mode="updates"):
    print(chunk)

# Inspect the final state snapshot for this thread.
final_state = assistant.get_state(config)
print("\napproved:", final_state.values["approved"])
print("最後一則訊息:", final_state.values["messages"][-1].content)
# Expected output (大致如下):
# === 第二段：核准後續跑 ===
# {'human_approve': {'approved': True, 'messages': [ToolMessage(content='已下單：...')]}}
#
# approved: True
# 最後一則訊息: 已下單：VIP 限定杯子 x 2，訂單編號 ORD-2026-001

## 🧪 練習 1：把 Capstone 改成 `values` 模式觀察

把第 12 格的 `stream_mode="updates"` 改成 `"values"`，重跑一次（記得換一個新的
`thread_id`，例如 `"capstone-2"`）。

觀察：
- 每個 chunk 是不是變成**完整 state 快照**（同時看得到 messages / context / approved）？
- 對比 `updates`，哪一種比較適合「除錯狀態」、哪一種比較適合「做進度條」？

In [ ]:
# 在這裡寫你的答案
# config2 = {"configurable": {"thread_id": "capstone-2"}}
# for chunk in assistant.stream(user_turn, config2, stream_mode="values"):
#     print(chunk)

## 🧪 練習 2：拒絕核准的分支

重跑第 12 格（用新的 `thread_id`）讓圖停在核准點，然後在第 13 格改成
`Command(resume="no")` 續跑。

觀察：
- `place_order` 這次有沒有被執行？（看最後的 messages 有沒有 ToolMessage）
- `approved` 最後是 `True` 還是 `False`？
- 想一想：這正是「危險工具呼叫前先問人」的安全閥——人說不，外部狀態就不會被改動。

In [ ]:
# 在這裡寫你的答案

## 小結 & 下一步

這一格回顧本模組，也收束整套課程：

- **串流**：同一張圖、同一個輸入，換 `stream_mode` 換觀看角度——
  `"values"` 看完整 state 快照、`"updates"` 看每步增量、`"messages"` 看 token 級逐字輸出。
  `stream` 不改變圖的邏輯，只改變輸出方式。
- **可觀測**：設好 `LANGSMITH_TRACING` / `LANGSMITH_API_KEY` 兩個環境變數，
  執行就自動上傳 trace，在 UI 上把每個節點的輸入輸出、模型呼叫、token 用量攤平成一棵樹。
- **Capstone**：一張圖把全課程疊起來——`retrieve`(RAG, 第一冊 M05) →
  `agent`(工具, M04) → `human_approve`(interrupt, 第二冊 M05)，
  compile 掛 `checkpointer`(第二冊 M04) 取得記憶，最後用 `stream` 逐步跑完。

**整套學習地圖**：
Runnable 負責「一步怎麼算」（第一冊），Graph 負責「多步怎麼走」（第二冊），
stream 與 LangSmith 負責「讓你看見它在走」（本模組）。

**延伸方向**（課程之外）：
- 部署：把圖包成 API（FastAPI + `graph.astream`），或用 LangGraph Platform 一鍵部署。
- 持久化升級：`InMemorySaver` → `SqliteSaver` / Postgres，記憶跨重啟存活。
- 評估：用 LangSmith 的 dataset + evaluator 對圖做回歸測試，量化改動是變好還是變壞。

恭喜——你已具備從零搭建、編排、觀測一個 LLM 應用的完整能力。